### Overview
In this notebook, we will compute the performance of our retrieve agent i.e, given a query, computing the accuracy of how many times the top k columns contains the target columns (required to answer the query)
- Accuracy is crucial because if we miss a column that is important to answer the question, then further reACT agent wont be able to analyze it
- This common framework of retrieving the important column can help us when there are too many columns in the database (Part 2 in this challenge)

### Load evaluation dataset

In [1]:
import pandas as pd
eval_data = pd.read_csv("./data/retriever_evaluation_data.csv")
eval_data['target'] = eval_data['columns'].apply(eval)
eval_data.drop(['columns'], axis=1, inplace=True)
print(eval_data.shape)
eval_data.head(2)

(106, 2)


,question,target
0,How many unique source system types are present?,[SOURCE_SYSTEM_TYPE]
1,What is the average fire size across all incid...,[FIRE_SIZE]


In [2]:
# complexity of queries - 1 means less complex, 5 - highly complex
eval_data['target'].apply(len).value_counts()

target
1    41
2    34
3    14
4    11
5     6
Name: count, dtype: int64

In [3]:
import sys
import os
sys.path.append("/project/pi_hongyu_umass_edu/zonghai/clinical-llm-alignment/durga_sandeep/Aira/submission/helper/")
sys.path.append("../")

# import the retrieval agent
from retrieve_top_k_columns import RetrievalAgent
retrieve_agent = RetrievalAgent()

Agent Configurations
Model :  gpt-3.5-turbo
k (no. of columns to extract):  5


In [4]:
# sample example run
retrieve_agent.run("what are the average fire size in each county?")

['FIRE_SIZE',
 'FIRE_SIZE_CLASS',
 'COUNTY',
 'STATE',
 'FIPS_CODE',
 'SOURCE_REPORTING_UNIT_NAME',
 'SOURCE_SYSTEM']

In [5]:
# 1 sec per query - this cell takes around 2-3 mins to execute
all_outputs = []
errors = []
for i in range(len(eval_data)):
    if i%10 == 0:
        print(f"{i} samples are processed")
    all_outputs.append(retrieve_agent.run(eval_data.iloc[i]['question']))

0 samples are processed
10 samples are processed
20 samples are processed
30 samples are processed
40 samples are processed
50 samples are processed
60 samples are processed
70 samples are processed
80 samples are processed
90 samples are processed
100 samples are processed


In [6]:
eval_data['predictions'] = all_outputs
eval_data.head()

,question,target,predictions
0,How many unique source system types are present?,[SOURCE_SYSTEM_TYPE],"[SOURCE_SYSTEM_TYPE, FOD_ID, FPA_ID, SOURCE_SY..."
1,What is the average fire size across all incid...,[FIRE_SIZE],"[FIRE_SIZE, FIRE_SIZE_CLASS, FIRE_YEAR, STATE,..."
2,What is the total number of incidents reported...,[SOURCE_SYSTEM],"[SOURCE_SYSTEM_TYPE, SOURCE_SYSTEM, FIRE_YEAR,..."
3,How many unique reporting agencies are present...,[NWCG_REPORTING_AGENCY],"[NWCG_REPORTING_AGENCY, NWCG_REPORTING_UNIT_ID..."
4,What is the distribution of fire causes (stati...,[STAT_CAUSE_DESCR],"[STAT_CAUSE_DESCR, STAT_CAUSE_CODE, FIRE_YEAR,..."


In [9]:
## performance -
accuracy = eval_data.apply(lambda x: set(x['target']) - set(x['predictions']), 1).apply(len).value_counts(normalize=True)[0]
print("Accuracy : ", accuracy)


Accuracy :  0.9905660377358491


In [10]:
eval_data['predictions'].apply(len).value_counts() # out of 39 columns present in the data - we brought it down to 5-7 columns

predictions
7    57
6    30
4    17
5     2
Name: count, dtype: int64

### Conclusion:
1. This framework can help us deal with very high dimensional data
2. In this example, we are able perform 99+% accurate by just utlizing 5-7 columns i.e., 12% of the total columns
3. Note that when we have higher number of columns we can increase our k-value to be more accurate, generally we might need 5-10% of columns for higher accuracy